# 🚛 EcoFleet AI — ML & Analytics Module
### FAR AWAY 2026 Hackathon | Logistics & Transit Theme
**Member 4 (ML/Data) Deliverable**

---
This notebook covers:
1. Mock Dataset Generation (realistic Indian logistics data)
2. Empty Mile Analytics
3. Carbon & Fuel Savings Calculator
4. Demand Forecasting (Linear Regression + Random Forest)
5. Shipment Matching Engine
6. All Visualizations
7. Final Hackathon Impact Metrics

## STEP 1: Install & Import Everything

In [ ]:
# Install required libraries (run this first in Colab)
# !pip install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set visual style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('✅ All libraries imported successfully!')
print('✅ EcoFleet AI ML Module — Ready to run')

## STEP 2: Generate Realistic Dataset
> **Why mock data?** We don't have a real company dataset, so we generate realistic data based on actual Indian logistics patterns. This is completely acceptable in hackathons.

In [ ]:
# --- CONFIGURATION ---
# Constants based on Indian logistics averages
MILEAGE_KM_PER_LITRE = 4.0       # Avg diesel truck mileage
DIESEL_COST_PER_LITRE = 90.0     # ₹90 per litre
CO2_KG_PER_LITRE = 2.68          # kg of CO2 per litre of diesel
RATE_PER_TON_PER_KM = 3.5        # ₹ per ton per km (market rate)

np.random.seed(42)  # For reproducibility

# Indian city pairs with realistic distances (km)
ROUTE_DATA = {
    ('Chennai', 'Bangalore'): 350,
    ('Bangalore', 'Hyderabad'): 570,
    ('Chennai', 'Hyderabad'): 630,
    ('Bangalore', 'Vellore'): 210,
    ('Hyderabad', 'Pune'): 560,
    ('Mumbai', 'Pune'): 150,
    ('Chennai', 'Coimbatore'): 500,
    ('Bangalore', 'Mysore'): 140,
    ('Pune', 'Mumbai'): 150,
    ('Hyderabad', 'Nagpur'): 500,
    ('Delhi', 'Jaipur'): 280,
    ('Delhi', 'Lucknow'): 555,
    ('Mumbai', 'Surat'): 284,
    ('Kolkata', 'Bhubaneswar'): 440,
    ('Chennai', 'Vellore'): 145,
}

routes = list(ROUTE_DATA.keys())
n_trips = 300
data = []

for i in range(n_trips):
    route = routes[np.random.randint(len(routes))]
    origin, destination = route
    distance_km = ROUTE_DATA[route] + np.random.randint(-20, 20)
    
    capacity_tons = np.random.choice([10, 15, 20, 25, 30])
    # 30% of trips are underloaded (the empty miles problem)
    if np.random.random() < 0.30:
        load_tons = round(np.random.uniform(0.1, 0.45) * capacity_tons, 1)
    else:
        load_tons = round(np.random.uniform(0.6, 0.95) * capacity_tons, 1)

    utilization = load_tons / capacity_tons
    empty_tons = capacity_tons - load_tons
    
    # Fuel: proportional to weight + base consumption
    fuel_consumed = round(distance_km / MILEAGE_KM_PER_LITRE * (0.7 + 0.3 * utilization), 1)
    fuel_cost = round(fuel_consumed * DIESEL_COST_PER_LITRE, 2)
    co2_emitted = round(fuel_consumed * CO2_KG_PER_LITRE, 2)
    
    # Revenue: only on actual loaded goods
    revenue = round(load_tons * distance_km * RATE_PER_TON_PER_KM, 2)
    # Potential revenue if truck was full
    potential_revenue = round(capacity_tons * distance_km * RATE_PER_TON_PER_KM, 2)
    revenue_lost = round(potential_revenue - revenue, 2)
    
    # Week number for time series
    week = (i // 6) + 1
    is_empty = 1 if utilization < 0.5 else 0
    
    data.append([
        i+1, origin, destination, distance_km,
        capacity_tons, load_tons, round(utilization*100, 1),
        empty_tons, fuel_consumed, fuel_cost,
        co2_emitted, revenue, potential_revenue, revenue_lost,
        week, is_empty
    ])

df = pd.DataFrame(data, columns=[
    'trip_id', 'origin', 'destination', 'distance_km',
    'capacity_tons', 'load_tons', 'utilization_pct',
    'empty_capacity_tons', 'fuel_consumed_L', 'fuel_cost_inr',
    'co2_emitted_kg', 'revenue_inr', 'potential_revenue_inr',
    'revenue_lost_inr', 'week', 'is_underloaded'
])

# Save to CSV
df.to_csv('ecofleet_shipments.csv', index=False)

print(f'✅ Dataset created: {len(df)} trips')
print(f'📦 Columns: {list(df.columns)}')
print(f'\n--- Dataset Preview ---')
df.head(5)

## STEP 3: Empty Mile Analytics
> The core problem. We calculate how much fuel, money, and CO2 is being wasted due to underloaded trucks.

In [ ]:
# --- EMPTY MILE ANALYTICS ---
empty_df = df[df['is_underloaded'] == 1]
full_df  = df[df['is_underloaded'] == 0]

total_trips       = len(df)
empty_trips       = len(empty_df)
empty_pct         = round(empty_trips / total_trips * 100, 1)
total_fuel_wasted = round(empty_df['fuel_consumed_L'].sum(), 1)
total_cost_wasted = round(empty_df['fuel_cost_inr'].sum(), 2)
total_co2_wasted  = round(empty_df['co2_emitted_kg'].sum(), 1)
total_rev_lost    = round(empty_df['revenue_lost_inr'].sum(), 2)
avg_util_empty    = round(empty_df['utilization_pct'].mean(), 1)
avg_util_full     = round(full_df['utilization_pct'].mean(), 1)

print('=' * 55)
print('       EMPTY MILE ANALYTICS — ECOFLEET AI')
print('=' * 55)
print(f'  Total trips analysed      : {total_trips}')
print(f'  Underloaded trips         : {empty_trips} ({empty_pct}%)')
print(f'  Fuel wasted               : {total_fuel_wasted:,.0f} Litres')
print(f'  Fuel cost wasted          : ₹{total_cost_wasted:,.0f}')
print(f'  CO₂ emitted (wasted)      : {total_co2_wasted:,.0f} kg')
print(f'  Revenue lost              : ₹{total_rev_lost:,.0f}')
print(f'  Avg utilisation (empty)   : {avg_util_empty}%')
print(f'  Avg utilisation (loaded)  : {avg_util_full}%')
print('=' * 55)

In [ ]:
# --- VISUALISATION 1: Empty vs Loaded Trips ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Empty Mile Problem — EcoFleet AI Analysis', fontsize=14, fontweight='bold')

# Pie chart: trip proportion
axes[0].pie(
    [empty_trips, total_trips - empty_trips],
    labels=['Underloaded', 'Fully Loaded'],
    colors=['#E24B4A', '#1D9E75'],
    autopct='%1.1f%%', startangle=90
)
axes[0].set_title('Trip Distribution')

# Bar: utilization comparison
axes[1].bar(['Underloaded\nTrips', 'Loaded\nTrips'],
            [avg_util_empty, avg_util_full],
            color=['#E24B4A', '#1D9E75'], width=0.5)
axes[1].set_ylabel('Avg Utilisation (%)')
axes[1].set_title('Utilisation Comparison')
axes[1].set_ylim(0, 100)
for i, v in enumerate([avg_util_empty, avg_util_full]):
    axes[1].text(i, v + 1, f'{v}%', ha='center', fontweight='bold')

# Histogram: utilization distribution
axes[2].hist(df['utilization_pct'], bins=20, color='#185FA5', edgecolor='white')
axes[2].axvline(50, color='#E24B4A', linestyle='--', label='50% threshold')
axes[2].set_xlabel('Utilisation (%)')
axes[2].set_ylabel('Number of Trips')
axes[2].set_title('Utilisation Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig('chart1_empty_miles.png', bbox_inches='tight')
plt.show()
print('✅ Chart 1 saved: chart1_empty_miles.png')

## STEP 4: Carbon & Fuel Savings Calculator
> If EcoFleet AI fills those empty trucks — how much do we save?

In [ ]:
# --- CARBON SAVINGS CALCULATOR ---
# Assumption: EcoFleet AI fills underloaded trucks to 80% utilisation
TARGET_UTILISATION = 0.80

# For each underloaded trip, calculate what COULD be saved
empty_df = empty_df.copy()
empty_df['optimised_load'] = empty_df['capacity_tons'] * TARGET_UTILISATION
empty_df['extra_load_tons'] = empty_df['optimised_load'] - empty_df['load_tons']

# Fuel saved = fuel used for empty space portion
empty_df['fuel_saved_L'] = (
    (empty_df['capacity_tons'] - empty_df['load_tons']) /
    empty_df['capacity_tons'] * 0.30 *
    empty_df['fuel_consumed_L']
).round(2)

empty_df['money_saved_inr'] = (empty_df['fuel_saved_L'] * DIESEL_COST_PER_LITRE).round(2)
empty_df['co2_saved_kg']    = (empty_df['fuel_saved_L'] * CO2_KG_PER_LITRE).round(2)
empty_df['extra_revenue']   = (
    empty_df['extra_load_tons'] * empty_df['distance_km'] * RATE_PER_TON_PER_KM
).round(2)

# Totals
total_fuel_saved  = round(empty_df['fuel_saved_L'].sum(), 0)
total_money_saved = round(empty_df['money_saved_inr'].sum(), 0)
total_co2_saved   = round(empty_df['co2_saved_kg'].sum(), 0)
total_extra_rev   = round(empty_df['extra_revenue'].sum(), 0)

print('=' * 55)
print('   CARBON & FUEL SAVINGS — IF ECOFLEET AI IS USED')
print('=' * 55)
print(f'  Fuel Saved         : {total_fuel_saved:,.0f} Litres')
print(f'  Money Saved        : ₹{total_money_saved:,.0f}')
print(f'  CO₂ Reduced        : {total_co2_saved:,.0f} kg')
print(f'  Extra Revenue      : ₹{total_extra_rev:,.0f}')
print(f'  Trips Optimised    : {len(empty_df)}')
print('=' * 55)
print('\n💡 These numbers go directly into your presentation slides!')

In [ ]:
# --- VISUALISATION 2: Savings Impact ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('EcoFleet AI — Projected Savings Impact', fontsize=14, fontweight='bold')

# Bar: metrics comparison (before vs after)
categories = ['Fuel\n(Litres)', 'CO₂\n(kg)', 'Revenue\n(₹000s)']
before_vals = [
    round(total_fuel_wasted),
    round(total_co2_wasted),
    round(total_rev_lost / 1000)
]
after_vals = [
    round(total_fuel_wasted - total_fuel_saved),
    round(total_co2_wasted - total_co2_saved),
    round((total_rev_lost - total_extra_rev) / 1000)
]

x = np.arange(len(categories))
w = 0.35
axes[0].bar(x - w/2, before_vals, w, label='Before EcoFleet', color='#E24B4A')
axes[0].bar(x + w/2, after_vals,  w, label='After EcoFleet',  color='#1D9E75')
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories)
axes[0].set_title('Before vs After Optimisation')
axes[0].legend()
axes[0].set_ylabel('Value')

# Horizontal bar: savings summary
labels = ['Fuel Saved (L)', 'CO₂ Saved (kg)', 'Extra Revenue (₹)']
values = [total_fuel_saved, total_co2_saved, total_extra_rev / 10000]
colors = ['#185FA5', '#1D9E75', '#BA7517']
bars = axes[1].barh(labels, values, color=colors)
axes[1].set_title('Total Savings from EcoFleet AI')
axes[1].set_xlabel('Value (Revenue scaled ÷ 10,000)')
for bar, val in zip(bars, [total_fuel_saved, total_co2_saved, total_extra_rev]):
    axes[1].text(bar.get_width() * 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:,.0f}', va='center', fontweight='bold', color='white')

plt.tight_layout()
plt.savefig('chart2_savings.png', bbox_inches='tight')
plt.show()
print('✅ Chart 2 saved: chart2_savings.png')

## STEP 5: Demand Forecasting (ML Model)
> Predict future shipment demand by week using Linear Regression and Random Forest.

In [ ]:
# --- DEMAND FORECASTING ---
# Aggregate shipments per week
weekly = df.groupby('week').agg(
    total_shipments=('trip_id', 'count'),
    avg_utilisation=('utilization_pct', 'mean'),
    total_revenue=('revenue_inr', 'sum'),
    total_co2=('co2_emitted_kg', 'sum')
).reset_index()

# Add time features
weekly['week_sq'] = weekly['week'] ** 2

X = weekly[['week', 'week_sq']].values
y = weekly['total_shipments'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- MODEL 1: Linear Regression ---
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)

# --- MODEL 2: Random Forest ---
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf  = r2_score(y_test, y_pred_rf)

# Forecast next 4 weeks
last_week = weekly['week'].max()
future_weeks = np.array([[last_week+i, (last_week+i)**2] for i in range(1, 5)])
future_pred_lr = lr.predict(future_weeks)
future_pred_rf = rf.predict(future_weeks)

print('=' * 50)
print('        DEMAND FORECASTING RESULTS')
print('=' * 50)
print(f'Linear Regression  — MAE: {mae_lr:.2f} | R²: {r2_lr:.3f}')
print(f'Random Forest      — MAE: {mae_rf:.2f} | R²: {r2_rf:.3f}')
print(f'\nForecast (next 4 weeks — Random Forest):')
for i, pred in enumerate(future_pred_rf):
    print(f'  Week {last_week+i+1}: {int(pred)} shipments predicted')
print('=' * 50)

In [ ]:
# --- VISUALISATION 3: Demand Forecast ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Shipment Demand Forecasting — EcoFleet AI', fontsize=14, fontweight='bold')

# Forecast chart
future_week_nums = [last_week + i for i in range(1, 5)]
axes[0].plot(weekly['week'], weekly['total_shipments'],
             'o-', color='#185FA5', label='Historical', linewidth=2)
axes[0].plot(future_week_nums, future_pred_rf,
             's--', color='#1D9E75', label='RF Forecast', linewidth=2)
axes[0].plot(future_week_nums, future_pred_lr,
             '^--', color='#BA7517', label='LR Forecast', linewidth=2)
axes[0].axvline(last_week, color='gray', linestyle=':', alpha=0.7, label='Forecast start')
axes[0].set_xlabel('Week')
axes[0].set_ylabel('Number of Shipments')
axes[0].set_title('Weekly Shipment Demand + Forecast')
axes[0].legend()

# Model comparison bar
models = ['Linear Regression', 'Random Forest']
r2_scores = [max(0, r2_lr), max(0, r2_rf)]
bars = axes[1].bar(models, r2_scores, color=['#BA7517', '#1D9E75'], width=0.4)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('R² Score (higher = better)')
axes[1].set_title('Model Accuracy Comparison')
for bar, val in zip(bars, r2_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('chart3_forecast.png', bbox_inches='tight')
plt.show()
print('✅ Chart 3 saved: chart3_forecast.png')

## STEP 6: Shipment Matching Engine
> The core AI feature. Given a truck's details, find the best matching shipments.

In [ ]:
# --- SHIPMENT MATCHING ENGINE ---
# Constants
MILEAGE_KM_PER_LITRE = 4.0
DIESEL_COST_PER_LITRE = 90.0
CO2_KG_PER_LITRE = 2.68
RATE_PER_TON_PER_KM = 3.5

def calculate_match_score(truck, shipment):
    """
    Scores how well a shipment matches a truck.
    Returns a score from 0–100.
    """
    score = 0
    reasons = []

    # 1. CAPACITY CHECK (40 points)
    free_capacity = truck['capacity_tons'] - truck['current_load_tons']
    if shipment['weight_tons'] <= free_capacity:
        utilisation_gain = shipment['weight_tons'] / free_capacity
        score += int(40 * utilisation_gain)
        reasons.append(f'Weight fits (+{int(40 * utilisation_gain)} pts)')
    else:
        reasons.append('Weight exceeds capacity (0 pts)')
        return 0, reasons  # Hard fail

    # 2. ROUTE COMPATIBILITY (35 points)
    if shipment['pickup'] == truck['current_destination']:
        score += 35
        reasons.append('Pickup = truck destination (+35 pts)')
    elif shipment['pickup'] in truck['route_cities']:
        score += 20
        reasons.append('Pickup is on route (+20 pts)')
    else:
        score += 5
        reasons.append('Pickup is off route (+5 pts)')

    # 3. DISTANCE EFFICIENCY (25 points)
    detour_penalty = abs(shipment['distance_km'] - truck['remaining_km'])
    distance_score = max(0, 25 - int(detour_penalty / 20))
    score += distance_score
    reasons.append(f'Distance efficiency (+{distance_score} pts)')

    return min(score, 100), reasons


def find_best_matches(truck, shipments, top_n=3):
    """
    Returns the top N best matching shipments for a truck.
    """
    results = []
    for s in shipments:
        score, reasons = calculate_match_score(truck, s)
        if score > 0:
            free_cap = truck['capacity_tons'] - truck['current_load_tons']
            fuel_saved = (s['distance_km'] / MILEAGE_KM_PER_LITRE) * CO2_KG_PER_LITRE
            extra_rev  = s['weight_tons'] * s['distance_km'] * RATE_PER_TON_PER_KM
            results.append({
                'shipment_id'   : s['id'],
                'pickup'        : s['pickup'],
                'delivery'      : s['delivery'],
                'weight_tons'   : s['weight_tons'],
                'distance_km'   : s['distance_km'],
                'match_score'   : score,
                'co2_saved_kg'  : round(fuel_saved, 1),
                'extra_rev_inr' : round(extra_rev, 0),
                'reasons'       : reasons
            })
    results.sort(key=lambda x: x['match_score'], reverse=True)
    return results[:top_n]


# ── DEMO: The exact scenario from the project brief ──
truck = {
    'id'                   : 'TN01-AB-1234',
    'current_destination'  : 'Bangalore',
    'capacity_tons'        : 20,
    'current_load_tons'    : 14,
    'remaining_km'         : 350,
    'route_cities'         : ['Chennai', 'Bangalore', 'Vellore']
}

available_shipments = [
    {'id': 'SHP-001', 'pickup': 'Bangalore', 'delivery': 'Vellore',
     'weight_tons': 5, 'distance_km': 210},
    {'id': 'SHP-002', 'pickup': 'Bangalore', 'delivery': 'Mysore',
     'weight_tons': 3, 'distance_km': 140},
    {'id': 'SHP-003', 'pickup': 'Hyderabad', 'delivery': 'Pune',
     'weight_tons': 8, 'distance_km': 560},
    {'id': 'SHP-004', 'pickup': 'Bangalore', 'delivery': 'Coimbatore',
     'weight_tons': 2, 'distance_km': 360},
    {'id': 'SHP-005', 'pickup': 'Chennai', 'delivery': 'Vellore',
     'weight_tons': 10, 'distance_km': 145},
]

matches = find_best_matches(truck, available_shipments, top_n=3)

print('=' * 60)
print('  SHIPMENT MATCHING ENGINE — ECOFLEET AI')
print('=' * 60)
print(f'  Truck      : {truck["id"]}')
print(f'  Route      : Chennai → Bangalore')
print(f'  Capacity   : {truck["capacity_tons"]} tons | Loaded: {truck["current_load_tons"]} tons')
print(f'  Free Space : {truck["capacity_tons"] - truck["current_load_tons"]} tons')
print('=' * 60)
print('  TOP MATCHES FOUND:')
print('-' * 60)
for i, m in enumerate(matches, 1):
    print(f'  #{i} [{m["shipment_id"]}] {m["pickup"]} → {m["delivery"]}')
    print(f'      Weight: {m["weight_tons"]}t | Distance: {m["distance_km"]}km')
    print(f'      Match Score : {m["match_score"]}/100')
    print(f'      CO₂ Saved   : {m["co2_saved_kg"]} kg')
    print(f'      Extra Rev   : ₹{m["extra_rev_inr"]:,.0f}')
    print(f'      Reasons     : {" | ".join(m["reasons"])}')
    print()

## STEP 7: Route & Revenue Visualisations

In [ ]:
# --- VISUALISATION 4: Route Analysis & Revenue ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('EcoFleet AI — Route & Revenue Analytics', fontsize=14, fontweight='bold')

# Top routes by volume
df['route'] = df['origin'] + ' → ' + df['destination']
top_routes = df['route'].value_counts().head(8)
axes[0,0].barh(top_routes.index, top_routes.values, color='#185FA5')
axes[0,0].set_title('Top Routes by Trip Volume')
axes[0,0].set_xlabel('Number of Trips')

# Revenue per route (top 6)
rev_by_route = df.groupby('route')['revenue_inr'].sum().nlargest(6)
axes[0,1].bar(range(len(rev_by_route)), rev_by_route.values, color='#1D9E75')
axes[0,1].set_xticks(range(len(rev_by_route)))
axes[0,1].set_xticklabels(rev_by_route.index, rotation=30, ha='right', fontsize=9)
axes[0,1].set_title('Revenue by Route (₹)')
axes[0,1].set_ylabel('Total Revenue (₹)')

# Weekly revenue trend
weekly_rev = df.groupby('week')['revenue_inr'].sum()
axes[1,0].fill_between(weekly_rev.index, weekly_rev.values, alpha=0.3, color='#185FA5')
axes[1,0].plot(weekly_rev.index, weekly_rev.values, 'o-', color='#185FA5', linewidth=2)
axes[1,0].set_title('Weekly Revenue Trend')
axes[1,0].set_xlabel('Week')
axes[1,0].set_ylabel('Revenue (₹)')

# Matching engine: score bar chart
match_ids = [m['shipment_id'] for m in matches]
match_scores = [m['match_score'] for m in matches]
match_colors = ['#1D9E75' if s >= 70 else '#BA7517' if s >= 40 else '#E24B4A' for s in match_scores]
axes[1,1].bar(match_ids, match_scores, color=match_colors)
axes[1,1].set_ylim(0, 100)
axes[1,1].set_title('Shipment Match Scores')
axes[1,1].set_ylabel('Match Score (0-100)')
axes[1,1].axhline(70, color='green', linestyle='--', alpha=0.5, label='Good match (70+)')
axes[1,1].legend()
for i, (bar, val) in enumerate(zip(axes[1,1].patches, match_scores)):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                  f'{val}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('chart4_routes_revenue.png', bbox_inches='tight')
plt.show()
print('✅ Chart 4 saved: chart4_routes_revenue.png')

## STEP 8: Final Impact Metrics (For Slides)

In [ ]:
# --- FINAL HACKATHON IMPACT METRICS ---
empty_reduction_pct     = round((total_fuel_saved / total_fuel_wasted) * 100, 1)
utilisation_improvement = round(avg_util_full - avg_util_empty, 1)
trees_equivalent        = round(total_co2_saved / 21)  # 1 tree absorbs ~21kg CO2/year

print('\n')
print('╔══════════════════════════════════════════════════╗')
print('║      ECOFLEET AI — HACKATHON IMPACT METRICS     ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Empty trips reduced      : {empty_pct}% of all trips       ║')
print(f'║  Fuel saved               : {total_fuel_saved:,.0f} Litres           ║')
print(f'║  CO₂ reduced              : {total_co2_saved:,.0f} kg               ║')
print(f'║  Equivalent trees planted : {trees_equivalent} trees             ║')
print(f'║  Extra revenue unlocked   : ₹{total_extra_rev:,.0f}          ║')
print(f'║  Fuel cost saved          : ₹{total_money_saved:,.0f}          ║')
print(f'║  Utilisation improved by  : +{utilisation_improvement}%               ║')
print(f'║  Trips in dataset         : {total_trips}                      ║')
print('╚══════════════════════════════════════════════════╝')
print('\n💡 COPY THESE NUMBERS INTO YOUR PRESENTATION SLIDES!')
print('💡 Give fuel/CO2/revenue numbers to M1 immediately.')

In [ ]:
# --- VISUALISATION 5: Final Impact Dashboard ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('EcoFleet AI — Final Impact Dashboard', fontsize=14, fontweight='bold')

# Impact metrics as large text cards
metrics = [
    ('Fuel Saved', f'{total_fuel_saved:,.0f} L', '#185FA5'),
    ('CO₂ Reduced', f'{total_co2_saved:,.0f} kg', '#1D9E75'),
    ('Extra Revenue', f'₹{total_extra_rev/100000:.1f}L', '#BA7517'),
]
for ax, (label, value, color) in zip(axes, metrics):
    ax.set_facecolor(color + '15')
    ax.text(0.5, 0.65, value, transform=ax.transAxes,
            fontsize=28, fontweight='bold', color=color, ha='center')
    ax.text(0.5, 0.35, label, transform=ax.transAxes,
            fontsize=13, color='#444', ha='center')
    ax.text(0.5, 0.18, f'across {total_trips} trips', transform=ax.transAxes,
            fontsize=10, color='#888', ha='center')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)

plt.tight_layout()
plt.savefig('chart5_impact_dashboard.png', bbox_inches='tight')
plt.show()
print('✅ Chart 5 saved: chart5_impact_dashboard.png')
print('\n✅ ALL DONE! Files to commit to GitHub /ml/ folder:')
print('   ecofleet_shipments.csv')
print('   chart1_empty_miles.png')
print('   chart2_savings.png')
print('   chart3_forecast.png')
print('   chart4_routes_revenue.png')
print('   chart5_impact_dashboard.png')